# Stage 3 — Feature Selection (Mutual Information / ANOVA F-test)
### Cancer Prediction from Gene Expression Data — Prediction Track
### Binary Classification: Tumor vs Normal (merged multi-dataset breast cancer data)

This notebook loads the preprocessed, batch-corrected, balanced binary data from `01_preprocessing.ipynb`
and reduces the gene set down to the top 200–500 most informative genes, using:
- **Mutual Information** (primary method)
- **ANOVA F-test** (cross-check)

Label convention (must match notebook 01): **1 = tumor/cancer, 0 = normal**.

## 0. Setup

In [7]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif, f_classif
import joblib
import os

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

from google.colab import drive
drive.mount('/content/drive')

INPUT_DIR = "/content/drive/MyDrive/Gene Project/processed_data"
OUTPUT_DIR = "/content/drive/MyDrive/Gene Project/processed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Load Preprocessed Data

In [9]:
X_train = pd.read_csv(f"{INPUT_DIR}/X_train.csv")
X_test = pd.read_csv(f"{INPUT_DIR}/X_test.csv")
y_train = pd.read_csv(f"{INPUT_DIR}/y_train.csv").iloc[:, 0].values
y_test = pd.read_csv(f"{INPUT_DIR}/y_test.csv").iloc[:, 0].values
label_mapping = joblib.load(f"{INPUT_DIR}/label_mapping.pkl")

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Label mapping:", label_mapping)
print("Genes available going into feature selection:", X_train.shape[1])
print("Train class balance:", pd.Series(y_train).value_counts().to_dict())

X_train: (860, 16488)  X_test: (117, 16488)
Label mapping: {0: 'normal', 1: 'tumor'}
Genes available going into feature selection: 16488
Train class balance: {1: 430, 0: 430}


## 2. Mutual Information Feature Selection (Primary Method)
Scores every gene on how much information it provides about the tumor/normal label. Captures both linear
and non-linear associations.

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
N_FEATURES = 300   # tune this (typical range: 200-500)

mi_scores = mutual_info_classif(X_train, y_train, random_state=RANDOM_STATE)
mi_ranking = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

print("Top 15 genes by Mutual Information:")
mi_ranking.head(15)

Top 15 genes by Mutual Information:


,0
208190_s_at,0.448983
203435_s_at,0.418976
209186_at,0.411688
209326_at,0.405676
206742_at,0.404513
213451_x_at,0.403051
227419_x_at,0.401519
212858_at,0.398901
213715_s_at,0.395462
232510_s_at,0.394776


In [12]:
top_genes_mi = mi_ranking.head(N_FEATURES).index.tolist()
print(f"Selected {len(top_genes_mi)} genes using Mutual Information")

Selected 300 genes using Mutual Information


## 3. ANOVA F-test Feature Selection (Cross-check)
For binary classification, ANOVA F-test is essentially a two-sample comparison of mean expression between
tumor and normal groups per gene — a fast, well-understood cross-check against the MI ranking.

In [13]:
f_scores, p_values = f_classif(X_train, y_train)
anova_ranking = pd.Series(f_scores, index=X_train.columns).sort_values(ascending=False)

print("Top 15 genes by ANOVA F-test:")
anova_ranking.head(15)

Top 15 genes by ANOVA F-test:


,0
208190_s_at,1093.829647
209186_at,1075.098319
203228_at,1048.497999
227419_x_at,1041.796801
229839_at,1032.319161
203435_s_at,1031.705695
200644_at,981.062091
229357_at,959.479280
202503_s_at,957.949464
217755_at,950.233188


In [14]:
top_genes_anova = anova_ranking.head(N_FEATURES).index.tolist()

overlap = set(top_genes_mi) & set(top_genes_anova)
print(f"Genes selected by Mutual Information: {len(top_genes_mi)}")
print(f"Genes selected by ANOVA F-test: {len(top_genes_anova)}")
print(f"Overlap between the two methods: {len(overlap)} genes "
      f"({len(overlap)/N_FEATURES*100:.1f}% agreement)")

Genes selected by Mutual Information: 300
Genes selected by ANOVA F-test: 300
Overlap between the two methods: 229 genes (76.3% agreement)


## 4. Sanity Check Against Known Breast Cancer Genes
A quick, cheap credibility check: known breast-cancer-associated genes (e.g. ESR1, ERBB2/HER2, MKI67,
GATA3, FOXA1) should generally show up with high MI/ANOVA scores if the pipeline (probe ID matching, batch
correction, labeling) worked correctly. If none of these appear anywhere near the top, it's worth
re-checking the label rules from notebook 01 before trusting the results.

Note: GPL570 uses Affymetrix probe IDs (e.g. `205225_at` for ESR1), not plain gene symbols, so this check
requires a probe-to-gene-symbol mapping if your `X_train` columns are probe IDs rather than gene symbols.
If you haven't mapped probes to gene symbols yet, skip this cell or map a few known probe IDs manually.

In [15]:
# Example known breast-cancer probe IDs on GPL570 (Affymetrix HG-U133 Plus 2.0) — adjust/verify against
# your platform annotation file if your data uses different probe IDs
KNOWN_MARKER_PROBES = {
    "205225_at": "ESR1",
    "216836_s_at": "ERBB2 (HER2)",
    "212022_s_at": "MKI67",
    "209604_s_at": "GATA3",
    "204623_at": "FOXA1",
}

for probe, gene in KNOWN_MARKER_PROBES.items():
    if probe in mi_ranking.index:
        rank = list(mi_ranking.index).index(probe) + 1
        print(f"{gene} ({probe}): MI rank {rank} of {len(mi_ranking)}, score = {mi_ranking[probe]:.4f}")
    else:
        print(f"{gene} ({probe}): not found in current gene set (check probe ID / platform)")

ESR1 (205225_at): MI rank 1333 of 16488, score = 0.2370
ERBB2 (HER2) (216836_s_at): MI rank 1679 of 16488, score = 0.2234
MKI67 (212022_s_at): not found in current gene set (check probe ID / platform)
GATA3 (209604_s_at): MI rank 3190 of 16488, score = 0.1835
FOXA1 (204623_at): MI rank 4950 of 16488, score = 0.1527


## 5. Finalize Selected Gene Set
Using the Mutual Information ranking as the final feature set, consistent with the project architecture.

In [16]:
selected_genes = top_genes_mi

X_train_selected = X_train[selected_genes]
X_test_selected = X_test[selected_genes]

print("Final selected feature matrix:")
print("X_train_selected:", X_train_selected.shape)
print("X_test_selected:", X_test_selected.shape)

Final selected feature matrix:
X_train_selected: (860, 300)
X_test_selected: (117, 300)


## 6. Save Feature-Selected Data

In [17]:
X_train_selected.to_csv(f"{OUTPUT_DIR}/X_train_selected.csv", index=False)
X_test_selected.to_csv(f"{OUTPUT_DIR}/X_test_selected.csv", index=False)

joblib.dump(selected_genes, f"{OUTPUT_DIR}/selected_genes.pkl")
mi_ranking.to_csv(f"{OUTPUT_DIR}/mutual_information_scores.csv", header=["mi_score"])

print("Saved feature-selected data and gene list to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

Saved feature-selected data and gene list to: /content/drive/MyDrive/Gene Project/processed_data
['y_train.csv', 'scaler.pkl', 'X_train.csv', 'X_test.csv', 'variance_selector.pkl', 'y_test.csv', 'genes_after_variance_filter.pkl', 'label_mapping.pkl', 'X_train_selected.csv', 'X_test_selected.csv', 'selected_genes.pkl', 'mutual_information_scores.csv']
